In [1]:
!pip install -q open3d
!pip install -q torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.4.1+cu121.html
!pip install -q timm pyvista

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 92.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 87.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 108.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 kB 13.9 MB/s eta 0:00:00


In [2]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# --- MEMORY-SAFE CONFIGURATION ---
CONFIG = {
    "ROOT_3D": '/kaggle/input/datasets/klein2111/scannet-3d/scannet_3d/train',
    "ROOT_2D": '/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d',
    
    # MEMORY FIXES:
    "NUM_POINTS": 32768,    # Halved the points (massively reduces RAM usage)
    "BATCH_SIZE": 2,        # Halved the batch size to fit safely in the T4 GPU
    
    "NUM_CLASSES": 20,      
    "PRETRAIN_EPOCHS": 5,   
    "FINETUNE_EPOCHS": 40,  
    "LR_PRETRAIN": 0.001,   
    "LR_FINETUNE": 0.001,
    "DEVICE": torch.device("cuda" if torch.cuda.is_available() else "cpu")
}
print(f"Running on: {CONFIG['DEVICE']}")

Running on: cuda


In [3]:
class ScanNetAugmentedDataset(Dataset):
    def __init__(self, root_3d, root_2d, num_points=65536, augment=True):
        self.root_3d = root_3d
        self.root_2d = root_2d
        self.num_points = num_points
        self.augment = augment
        self.file_list = sorted([f for f in os.listdir(root_3d) if f.endswith('.pth')])

    def __len__(self):
        return len(self.file_list)
        
    def apply_augmentation(self, points):
        theta = np.random.uniform(0, 2 * np.pi)
        rot_matrix = np.array([
            [np.cos(theta), -np.sin(theta), 0],
            [np.sin(theta),  np.cos(theta), 0],
            [0,              0,             1]
        ])
        points = np.dot(points, rot_matrix)
        noise = np.random.normal(0, 0.005, size=points.shape) # Reduced noise slightly for deep net
        points += noise
        return points

    def __getitem__(self, idx):
        file_name = self.file_list[idx]
        scene_id = file_name.replace('_vh_clean_2.pth', '')
        
        pc_path = os.path.join(self.root_3d, file_name)
        data_3d = torch.load(pc_path, weights_only=False)
        
        if isinstance(data_3d, tuple):
            points = data_3d[0]
            labels = data_3d[2] if len(data_3d) > 2 else data_3d[1] 
        elif isinstance(data_3d, dict):
            points = data_3d['coord']
            labels = data_3d.get('semantic_gt', data_3d.get('label', np.zeros(len(points))))
        else:
            points = data_3d[:, :3]
            labels = np.zeros(len(points))
            
        points = np.array(points) if torch.is_tensor(points) else points
        labels = np.array(labels) if torch.is_tensor(labels) else labels

        if len(points) >= self.num_points:
            s_idx = np.random.choice(len(points), self.num_points, replace=False)
        else:
            s_idx = np.random.choice(len(points), self.num_points, replace=True)
            
        points = points[s_idx]
        labels = labels[s_idx]
        
        if self.augment:
            points = self.apply_augmentation(points)
            
        img_path = os.path.join(self.root_2d, scene_id, "color", "0.jpg")
        if os.path.exists(img_path):
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, (518, 518))
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        else:
            image = torch.zeros((3, 518, 518))

        return {
            "points": torch.as_tensor(points).float(), 
            "labels": torch.as_tensor(labels).long(),
            "image": image
        }

In [4]:
def load_2d_teacher():
    model = timm.create_model('vit_small_patch14_dinov2', pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    model.eval()
    return model

class ConcertoStudentDeep(nn.Module):
    def __init__(self, output_dim=384):
        super().__init__()
        # Deep 1D Convolutional feature extractor
        self.conv1 = nn.Conv1d(3, 64, 1)
        self.conv2 = nn.Conv1d(64, 128, 1)
        self.conv3 = nn.Conv1d(128, 256, 1)
        self.conv4 = nn.Conv1d(256, 512, 1)
        self.conv5 = nn.Conv1d(512, 1024, 1)
        
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(128)
        self.bn3 = nn.BatchNorm1d(256)
        self.bn4 = nn.BatchNorm1d(512)
        self.bn5 = nn.BatchNorm1d(1024)

        # Projection specifically for Phase 1 (Synergy with DINOv2)
        self.synergy_proj = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, output_dim)
        )

    def forward(self, x):
        # 1D Convs expect shape [Batch, Channels, Points]
        x = x.permute(0, 2, 1)
        
        # Extract features at multiple depths
        x1 = F.relu(self.bn1(self.conv1(x)))
        x2 = F.relu(self.bn2(self.conv2(x1)))
        x3 = F.relu(self.bn3(self.conv3(x2)))
        x4 = F.relu(self.bn4(self.conv4(x3)))
        x5 = F.relu(self.bn5(self.conv5(x4))) # Output shape: [B, 1024, N]
        
        # Extract GLOBAL room feature (MaxPool over all N points)
        global_feat = torch.max(x5, 2, keepdim=True)[0] # Shape: [B, 1024, 1]
        
        # Map global feature to match DINOv2 2D teacher
        synergy_out = self.synergy_proj(global_feat.squeeze(2)) # [B, 384]
        
        # Return both the DINO mapping AND the deep geometric features for Phase 2
        return synergy_out, (x1, x2, x3, x4, x5, global_feat)

class SpatialSynergyNetDeep(nn.Module):
    def __init__(self, backbone, num_classes=20):
        super().__init__()
        self.backbone = backbone
        
        # CORRECTED MATH: 64 + 128 + 256 + 512 + 1024 + 1024 = 3008
        in_channels = 3008 
        
        self.seg_head = nn.Sequential(
            nn.Conv1d(in_channels, 1024, 1),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Conv1d(1024, 512, 1),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Conv1d(512, 256, 1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, num_classes, 1)
        )

    def forward(self, x):
        _, (x1, x2, x3, x4, x5, global_feat) = self.backbone(x)
        
        # Stretch the global room knowledge to every single point
        N = x.shape[1]
        global_feat_expanded = global_feat.repeat(1, 1, N) 
        
        # Combine local geometry with global room context
        concat_feat = torch.cat([x1, x2, x3, x4, x5, global_feat_expanded], dim=1) 
        
        # Classify
        out = self.seg_head(concat_feat) # [B, 20, N]
        return out.permute(0, 2, 1) # Return [B, N, 20] to match training loop

In [5]:
def run_massive_pipeline():
    dataset = ScanNetAugmentedDataset(CONFIG['ROOT_3D'], CONFIG['ROOT_2D'], augment=True)
    loader = DataLoader(dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=True)
    print(f"Data Loaded: {len(dataset)} scenes ready.")

    student = ConcertoStudentDeep().to(CONFIG['DEVICE'])
    teacher = load_2d_teacher().to(CONFIG['DEVICE'])
    
    # ========================================================
    # --- PHASE 1: PRE-TRAINING ---
    # ========================================================
    print("\n--- PHASE 1: PRE-TRAINING ---")
    optimizer_pre = optim.AdamW(student.parameters(), lr=CONFIG['LR_PRETRAIN'], weight_decay=1e-4)
    
    for epoch in range(CONFIG['PRETRAIN_EPOCHS']):
        student.train()
        total_loss = 0
        pbar = tqdm(loader, desc=f"Pretrain Epoch {epoch+1}")
        for batch in pbar:
            points, images = batch['points'].to(CONFIG['DEVICE']), batch['image'].to(CONFIG['DEVICE'])
            
            optimizer_pre.zero_grad()
            
            # The deep network returns the DINO projection as the first variable
            s_global, _ = student(points) 
            
            with torch.no_grad():
                t_feat = teacher(images)   
                
            loss = (1 - F.cosine_similarity(s_global, t_feat, dim=1)).mean()
            loss.backward()
            optimizer_pre.step()
            total_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    # FIX: We save the entire student state_dict directly now!
    torch.save(student.state_dict(), "backbone_massive.pth")
    print("Phase 1 Complete. Massive Backbone Saved.")

    # ========================================================
    # --- PHASE 2: FULL FINE-TUNING ---
    # ========================================================
    print("\n--- PHASE 2: DEEP SEGMENTATION FINE-TUNING ---")
    
    # We pass the entire 'student' object as the backbone
    final_model = SpatialSynergyNetDeep(student, num_classes=CONFIG['NUM_CLASSES']).to(CONFIG['DEVICE'])
    
    optimizer_full = optim.AdamW(final_model.parameters(), lr=CONFIG['LR_FINETUNE'], weight_decay=1e-4)
    
    # Standard weighted CrossEntropy works best with this deep architecture
    weights = torch.tensor([
        0.5, 0.5, 2.0, 2.0, 3.0, 2.5, 2.0, 1.5, 1.5, 2.5, 
        4.0, 3.0, 3.0, 2.5, 3.0, 4.0, 4.0, 4.0, 3.0, 1.0
    ]).to(CONFIG['DEVICE'])

    criterion = nn.CrossEntropyLoss(weight=weights, ignore_index=255)
    
    for epoch in range(CONFIG['FINETUNE_EPOCHS']):
        final_model.train()
        total_loss = 0
        pbar = tqdm(loader, desc=f"Finetune Epoch {epoch+1}")
        for batch in pbar:
            points, labels = batch['points'].to(CONFIG['DEVICE']), batch['labels'].to(CONFIG['DEVICE'])
            
            # Sanitize Labels
            labels[(labels < 0) | (labels >= CONFIG['NUM_CLASSES'])] = 255
            
            optimizer_full.zero_grad()
            
            # Forward pass and shape permutation for CrossEntropy
            preds = final_model(points).permute(0, 2, 1) 
            
            loss = criterion(preds, labels)
            loss.backward()
            optimizer_full.step()
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        print(f"Epoch {epoch+1} Avg Loss: {total_loss/len(loader):.4f}")

    # SAVE THE MASTER MODEL
    torch.save(final_model.state_dict(), "SpatialSynergyNet_Massive.pth")
    print("\nSUCCESS: Model saved as 'SpatialSynergyNet_Massive.pth'")

# Run the pipeline
run_massive_pipeline()

Data Loaded: 1201 scenes ready.


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]


--- PHASE 1: PRE-TRAINING ---


Pretrain Epoch 5: 100%|██████████| 601/601 [04:15<00:00,  2.35it/s, loss=0.6443]


Phase 1 Complete. Massive Backbone Saved.

--- PHASE 2: DEEP SEGMENTATION FINE-TUNING ---


...

Epoch 38 Avg Loss: 1.9926


Finetune Epoch 39: 100%|██████████| 601/601 [15:21<00:00,  1.53s/it, loss=2.0078]


Epoch 39 Avg Loss: 1.9922


Finetune Epoch 40: 100%|██████████| 601/601 [15:20<00:00,  1.53s/it, loss=2.4238]

Epoch 40 Avg Loss: 1.9850

SUCCESS: Model saved as 'SpatialSynergyNet_Massive.pth'
